# 02 - DuckDB Warehouse

This notebook creates a local DuckDB analytical warehouse for the Amsterdam Inside Airbnb dataset.

The purpose of this notebook is to load the cleaned listing master dataset into DuckDB and prepare SQL-based analytical tables for reporting, EDA, and future dashboarding.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.4


In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "amsterdam"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse"

WAREHOUSE_PATH.mkdir(parents=True, exist_ok=True)

DUCKDB_DATABASE_PATH = WAREHOUSE_PATH / "airbnb_market.duckdb"
LISTING_MASTER_FINAL_PATH = PROCESSED_DATA_PATH / "listing_master_final.parquet"

print("DuckDB database path:", DUCKDB_DATABASE_PATH)
print("Listing master path exists:", LISTING_MASTER_FINAL_PATH.exists())
print("Listing master path:", LISTING_MASTER_FINAL_PATH)

DuckDB database path: ..\warehouse\airbnb_market.duckdb
Listing master path exists: True
Listing master path: ..\data\processed\amsterdam\listing_master_final.parquet


In [3]:
conn = duckdb.connect(database=str(DUCKDB_DATABASE_PATH))

print("Connected to DuckDB successfully.")

Connected to DuckDB successfully.


In [4]:
conn.execute("""
CREATE OR REPLACE TABLE listing_master_final AS
SELECT *
FROM read_parquet(?)
""", [str(LISTING_MASTER_FINAL_PATH)])

print("listing_master_final table created successfully in DuckDB.")

listing_master_final table created successfully in DuckDB.


In [5]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,listing_master_final


In [6]:
conn.execute("""
SELECT 
    COUNT(*) AS row_count,
    COUNT(DISTINCT listing_id) AS unique_listing_ids
FROM listing_master_final
""").fetchdf()

,row_count,unique_listing_ids
0,10465,10465


In [7]:
conn.execute("""
SELECT 
    room_type,
    COUNT(*) AS listing_count,
    ROUND(AVG(price_numeric), 2) AS avg_price,
    ROUND(MEDIAN(price_numeric), 2) AS median_price,
    ROUND(AVG(availability_rate), 4) AS avg_availability_rate,
    ROUND(AVG(occupancy_proxy), 4) AS avg_occupancy_proxy
FROM listing_master_final
GROUP BY room_type
ORDER BY listing_count DESC
""").fetchdf()

,room_type,listing_count,avg_price,median_price,avg_availability_rate,avg_occupancy_proxy
0,Entire home/apt,8489,394.62,331.0,0.2133,0.7867
1,Private room,1929,203.67,171.0,0.4612,0.5388
2,Hotel room,26,249.85,227.5,0.7564,0.2436
3,Shared room,21,80.10,52.0,0.7769,0.2231


In [8]:
conn.close()

print("DuckDB connection closed.")

DuckDB connection closed.
